# Likestrømsmotor: startmotstand, hastighet og elektromekanisk dynamikk

## Pilotprosjekt for Matematikk 1

En elektrisk motor omformer elektrisk energi til mekanisk rotasjon. I dette prosjektet studerer vi en forenklet likestrømsmotor med konstant magnetfelt.

Når motoren står stille, er den motinduserte spenningen null. Strømmen kan derfor bli svært stor dersom ankermotstanden er liten. En klassisk løsning for større likestrømsmaskiner er en **seksjonert startmotstand** som gradvis kobles ut etter hvert som motoren får fart.

Prosjektet har tre deler:

1. **Lineær algebra:** dimensjonering av en seksjonert startmotstand
2. **Skalar ODE:** motorhastighet når ankerviklingens induktans neglisjeres
3. **Vektor-ODE:** koblet utvikling av strøm og vinkelhastighet

Prosjektet krever ikke vekselstrømsanalyse, impedans eller Laplace-transformasjon.

### Læringsmål

Etter prosjektet skal du kunne

- forklare hvorfor startstrømmen i en DC-motor kan bli stor,
- sette opp og løse et lineært system for motstandsseksjoner,
- beregne strøm, moment, motspenning og effekttap,
- utlede og løse en lineær førsteordens ODE,
- finne likevekt og tidskonstant i en motormodell,
- skrive den elektromekaniske modellen på matriseform,
- kontrollere at massematrisen er inverterbar,
- bruke Eulers metode på et koblet ODE-system,
- sammenligne direkte start, én startmotstand og seksjonert start.

### Modellforutsetninger

Vi antar blant annet konstant magnetfelt, lineær viskøs friksjon og konstante motorparametre. Startmotstanden er en klassisk resistiv startmetode. Moderne motordrifter bruker ofte elektronisk spenningsstyring, men den klassiske kretsen er velegnet til analyse med Ohms lov, lineær algebra og ODE-er.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

# Fysiske sammenhenger

Vi bruker disse sammenhengene gjennom prosjektet:

## Motindusert spenning

Når rotoren dreier, induseres en spenning som motvirker forsyningsspenningen:

$$
E_b=k_e\omega.
$$

Her er $\omega$ vinkelhastigheten og $k_e$ motorens spenningskonstant.

## Motormoment

For konstant magnetfelt er det elektromagnetiske momentet proporsjonalt med strømmen:

$$
T_m=k_t i.
$$

## Elektrisk krets

For en stasjonær eller induktansfri krets gjelder

$$
V=Ri+k_e\omega.
$$

Ved stillstand er $\omega=0$, og dermed

$$
i_{start}=\frac{V}{R}.
$$

Når $R$ er liten, kan startstrømmen bli stor.

# Del A: Seksjonert startmotstand

## A.1 Startprinsippet

Vi deler den eksterne startmotstanden i tre seksjoner:

```text
+V ---- R1 ---- R2 ---- R3 ---- Ra ---- motor ---- 0 V
        |       |       |
       S1      S2      S3
```

Bryterne $S_1,S_2,S_3$ kortslutter motstandsseksjonene gradvis. Ankermotstanden $R_a$ er alltid koblet inn.

De fire mulige totalmotstandene er

$$
\begin{aligned}
\mathcal R_1&=R_a+R_1+R_2+R_3,\\
\mathcal R_2&=R_a+R_2+R_3,\\
\mathcal R_3&=R_a+R_3,\\
\mathcal R_4&=R_a.
\end{aligned}
$$

Vi ønsker at strømmen skal holde seg mellom $I_{min}$ og $I_{max}$. Når strømmen har sunket til $I_{min}$, kortsluttes én seksjon, og strømmen hopper opp mot $I_{max}$.

## Oppgave A1: Første totalmotstand

Ved oppstart er $E_b=0$. Vis at totalmotstanden som gir startstrømmen $I_{max}$ er

$$
\boxed{\mathcal R_1=\frac{V}{I_{max}}.}
$$

Bruk

$$
V=48\ \mathrm V,\qquad
R_a=0.40\ \Omega,\qquad
I_{max}=20\ \mathrm A,\qquad
I_{min}=12\ \mathrm A.
$$

In [ ]:
V = 48.0
R_a = 0.40
I_max = 20.0
I_min = 12.0

Rtot1 = ...
print("Første totalmotstand:", Rtot1, "ohm")
print("Startstrøm:", ...)

## Oppgave A2: Totalmotstandene etter hvert trinn

Ved en kobling er hastigheten og dermed motspenningen den samme rett før og rett etter at en motstandsseksjon kobles ut.

Rett før koblingen:

$$
V-E_b=I_{min}\mathcal R_j.
$$

Rett etter koblingen:

$$
V-E_b=I_{max}\mathcal R_{j+1}.
$$

Vis derfor at

$$
\boxed{\mathcal R_{j+1}=q\mathcal R_j,\qquad
q=\frac{I_{min}}{I_{max}}.}
$$

Beregn $\mathcal R_2$ og $\mathcal R_3$. Sammenlign også den teoretiske $\mathcal R_4=q\mathcal R_3$ med den faktiske ankermotstanden $R_a$.

In [ ]:
q = I_min/I_max

Rtot2 = ...
Rtot3 = ...
Rtot4_teori = ...

print("q =", q)
print("Rtot1, Rtot2, Rtot3 =", Rtot1, Rtot2, Rtot3)
print("Teoretisk Rtot4 =", Rtot4_teori)
print("Faktisk Ra =", R_a)

## Oppgave A3: Lineært system for motstandsseksjonene

Motstandsseksjonene oppfyller

$$
\begin{aligned}
R_1+R_2+R_3&=\mathcal R_1-R_a,\\
R_2+R_3&=\mathcal R_2-R_a,\\
R_3&=\mathcal R_3-R_a.
\end{aligned}
$$

Skriv dette på formen

$$Ax=b$$

med $x=(R_1,R_2,R_3)^T$, og løs systemet med `np.linalg.solve`.

In [ ]:
A = np.array([
    [1.0, 1.0, 1.0],
    [0.0, 1.0, 1.0],
    [0.0, 0.0, 1.0]
])

b = np.array([
    Rtot1 - R_a,
    Rtot2 - R_a,
    Rtot3 - R_a
])

R_seksjoner = ...
R1, R2, R3 = R_seksjoner

print("R1, R2, R3 =", R_seksjoner)
print("Residual =", ...)

## Oppgave A4: Koblingshastigheter

Ved totalmotstanden $\mathcal R_j$ kobles neste seksjon ut når strømmen har falt til $I_{min}$:

$$
I_{min}=\frac{V-k_e\omega_j}{\mathcal R_j}.
$$

Dermed er koblingshastigheten

$$
\boxed{\omega_j=\frac{V-I_{min}\mathcal R_j}{k_e}.}
$$

Bruk $k_e=0.20\ \mathrm{V\,s/rad}$ og beregn koblingshastighetene for de tre totalmotstandene.

In [ ]:
k_e = 0.20
Rtot = np.array([Rtot1, Rtot2, Rtot3])

omega_kobling = ...
print("Koblingshastigheter i rad/s:", omega_kobling)
print("Koblingshastigheter i omdreininger/minutt:", ...)

## Oppgave A5: Kontroller strømmen før og etter kobling

For hver koblingshastighet skal strømmen være omtrent $I_{min}$ før koblingen og $I_{max}$ etter koblingen.

Kontroller dette numerisk. Ved den siste overgangen kobles $R_3$ ut slik at bare $R_a$ står igjen. På grunn av at $R_a$ ikke nødvendigvis er lik den teoretiske $\mathcal R_4$, blir det siste strømhoppet ikke nødvendigvis nøyaktig det samme som de andre.

In [ ]:
R_før = np.array([Rtot1, Rtot2, Rtot3])
R_etter = np.array([Rtot2, Rtot3, R_a])

I_før = ...
I_etter = ...

for j in range(3):
    print(j + 1, "før:", I_før[j], "A, etter:", I_etter[j], "A")

## Oppgave A6: Effekttap

Effekten som utvikles som varme i en motstand er

$$P_R=i^2R.$$

Beregn varmeeffekten i den eksterne startmotstanden ved begynnelsen av hvert trinn. Forklar hvorfor en resistiv starter er enkel, men lite energieffektiv.

In [ ]:
R_ekstern = R_før - R_a
P_start = ...

for j, Pj in enumerate(P_start, start=1):
    print(f"Trinn {j}: {Pj:.1f} W")

# Del B: Redusert motor uten induktans

Vi studerer nå en permanentmagnetisert DC-motor. Vi neglisjerer først ankerviklingens induktans. Strømmen tilpasser seg da momentant:

$$
i=\frac{V-k_e\omega}{R}.
$$

Rotorens bevegelseslikning er

$$
J\dot\omega=k_ti-b\omega-T_L.
$$

Her er

- $J$: treghetsmoment,
- $k_t$: momentkonstant,
- $b$: viskøs friksjonskoeffisient,
- $T_L$: konstant lastmoment.

## Oppgave B1: Utled den skalare ODE-en

Sett strømmen inn i bevegelseslikningen og vis at

$$
\boxed{
\dot\omega+a\omega=c,
}
$$

med

$$
a=\frac{k_tk_e}{JR}+\frac bJ,
\qquad
c=\frac{k_tV}{JR}-\frac{T_L}{J}.
$$

Finn likevektshastigheten

$$
\boxed{
\omega^*=\frac{c}{a}
=\frac{k_tV-RT_L}{k_tk_e+Rb}.
}
$$

Hva må gjelde for at $\omega^*>0$?

## Oppgave B2: Håndløsning

For konstant spenning og last er løsningen

$$
\omega(t)=\omega^*+\bigl(\omega_0-\omega^*\bigr)e^{-at}.
$$

Vis dette ved å løse ODE-en. Tidskonstanten er

$$
\tau_m=\frac1a.
$$

Bruk parameterne

$$
V=24\ \mathrm V,\quad R=0.80\ \Omega,\quad
k_e=k_t=0.12,\quad J=0.020,
$$

$$
b=0.002,\qquad T_L=0.20\ \mathrm{N\,m},
\qquad \omega(0)=0.
$$

In [ ]:
V_B = 24.0
R_B = 0.80
k_e_B = 0.12
k_t_B = 0.12
J_B = 0.020
b_B = 0.002
T_L_B = 0.20
omega0_B = 0.0

a_B = ...
c_B = ...
omega_likevekt_B = ...
tau_m = ...

print("Likevektshastighet:", omega_likevekt_B, "rad/s")
print("Mekanisk tidskonstant:", tau_m, "s")

## Oppgave B3: Euler-metoden

Implementer høyresiden og Eulers metode. Sammenlign den numeriske løsningen med håndløsningen.

In [ ]:
def redusert_motor(t, omega):
    strøm = (V_B - k_e_B*omega)/R_B
    domega = ...
    return domega


def euler_skalar(f, y0, T, h):
    N = int(round(T/h))
    t = np.linspace(0.0, N*h, N + 1)
    y = np.zeros(N + 1)
    y[0] = y0

    for n in range(N):
        y[n + 1] = ...

    return t, y


t_B, omega_B = euler_skalar(redusert_motor, omega0_B, T=5.0, h=0.005)
omega_eksakt_B = ...
strøm_B = ...

In [ ]:
fig, ax = plt.subplots(2, 1, sharex=True)

ax[0].plot(t_B, omega_B, label="Euler")
ax[0].plot(t_B, omega_eksakt_B, "--", label="Eksakt")
ax[0].set_ylabel("Vinkelhastighet rad/s")
ax[0].legend()
ax[0].grid()

ax[1].plot(t_B, strøm_B)
ax[1].set_xlabel("Tid s")
ax[1].set_ylabel("Strøm A")
ax[1].grid()

plt.show()

## Oppgave B4: Enkel startmotstand

La en ekstern motstand $R_s$ være innkoblet så lenge

$$\omega<\omega_s.$$

Da er

$$
R(\omega)=
\begin{cases}
R_a+R_s,&\omega<\omega_s,\\
R_a,&\omega\ge\omega_s.
\end{cases}
$$

Implementer modellen og sammenlign

- maksimal strøm,
- oppstartstid,
- tiden da motstanden kobles ut,
- energi tapt i startmotstanden.

In [ ]:
R_a_B = 0.40
R_start_B = 1.20
omega_s_B = 80.0


def redusert_motor_med_start(t, omega):
    R_tot = ...
    strøm = ...
    domega = ...
    return domega

# Simuler og etterberegn strøm og varmetap.

# Del C: Full elektromekanisk vektor-ODE

Nå tar vi med ankerviklingens induktans $L$. Den elektriske og mekaniske modellen er

$$
\boxed{
\begin{aligned}
L\dot i&=V-Ri-k_e\omega,\\
J\dot\omega&=k_ti-b\omega-T_L.
\end{aligned}}
$$

På massematriseform:

$$
\underbrace{
\begin{pmatrix}
L&0\\0&J
\end{pmatrix}}_{M}
\begin{pmatrix}
\dot i\\\dot\omega
\end{pmatrix}
=
\underbrace{
\begin{pmatrix}
-R&-k_e\\k_t&-b
\end{pmatrix}}_{K}
\begin{pmatrix}
i\\\omega
\end{pmatrix}
+
\begin{pmatrix}
V\\-T_L
\end{pmatrix}.
$$

## Oppgave C1: Er systemet en ODE?

Beregn

$$\det M=LJ.$$

Når $L>0$ og $J>0$, er $M$ inverterbar. Systemet kan derfor skrives eksplisitt som

$$
\boxed{
\begin{aligned}
\dot i&=\frac{V-Ri-k_e\omega}{L},\\
\dot\omega&=\frac{k_ti-b\omega-T_L}{J}.
\end{aligned}}
$$

Dette er et vanlig ODE-system, ikke en DAE.

Hva skjer dersom vi setter $L=0$? Forklar hvordan den algebraiske ligningen da kan løses for $i$ når $R>0$, slik at den reduserte modellen i del B kommer tilbake.

## Oppgave C2: Systemmatrise og stabilitet

Systemmatrisen er

$$
A=
\begin{pmatrix}
-R/L&-k_e/L\\
k_t/J&-b/J
\end{pmatrix}.
$$

1. Beregn egenverdiene.
2. Kontroller at sporet er negativt.
3. Kontroller at determinanten er positiv.
4. Hva sier egenverdienes realdeler om likevekten?
5. Sammenlign den elektriske tidskonstanten $L/R$ med de dynamiske tidsskalaene.

In [ ]:
L_C = 0.030
R_C = 0.80
k_e_C = 0.12
k_t_C = 0.12
J_C = 0.020
b_C = 0.002
V_C = 24.0
T_L_C = 0.20

M = np.array([
    [L_C, 0.0],
    [0.0, J_C]
])

K = np.array([
    [-R_C, -k_e_C],
    [k_t_C, -b_C]
])

A_C = ...
egenverdier = ...

print("det(M) =", ...)
print("A =
", A_C)
print("Egenverdier =", egenverdier)
print("Elektrisk tidskonstant L/R =", ...)

## Oppgave C3: Likevekt

Ved likevekt er $\dot i=\dot\omega=0$. Finn $i^*$ og $\omega^*$ ved å løse

$$
\begin{pmatrix}
R&k_e\\
k_t&-b
\end{pmatrix}
\begin{pmatrix}
i^*\\\omega^*
\end{pmatrix}
=
\begin{pmatrix}
V\\T_L
\end{pmatrix}.
$$

Kontroller likevekten ved å sette den inn i begge differensiallikningene.

In [ ]:
A_likevekt = np.array([
    [R_C, k_e_C],
    [k_t_C, -b_C]
])

b_likevekt = np.array([V_C, T_L_C])

x_likevekt = ...
print("i*, omega* =", x_likevekt)
print("Kontroll av høyresiden =", ...)

## Oppgave C4: Euler for strøm og hastighet

Bruk startverdiene

$$i(0)=0,\qquad\omega(0)=0.$$

Implementer systemet og Eulers metode.

In [ ]:
def full_motor(t, x):
    i, omega = x
    di = ...
    domega = ...
    return np.array([di, domega])


def euler_system(f, x0, T, h):
    N = int(round(T/h))
    t = np.linspace(0.0, N*h, N + 1)
    X = np.zeros((N + 1, len(x0)))
    X[0] = x0

    for n in range(N):
        X[n + 1] = ...

    return t, X


t_C, X_C = euler_system(
    full_motor,
    x0=np.array([0.0, 0.0]),
    T=5.0,
    h=0.0005
)

i_C = X_C[:, 0]
omega_C = X_C[:, 1]

In [ ]:
fig, ax = plt.subplots(2, 1, sharex=True)

ax[0].plot(t_C, i_C)
ax[0].axhline(x_likevekt[0], color="black", linestyle="--")
ax[0].set_ylabel("Strøm A")
ax[0].grid()

ax[1].plot(t_C, omega_C)
ax[1].axhline(x_likevekt[1], color="black", linestyle="--")
ax[1].set_xlabel("Tid s")
ax[1].set_ylabel("Vinkelhastighet rad/s")
ax[1].grid()

plt.show()

## Oppgave C5: Energi og effekt

Beregn

- elektrisk inngangseffekt
  $$P_{inn}=Vi,$$
- kobbertap
  $$P_R=i^2R,$$
- elektromekanisk omformet effekt
  $$P_{em}=k_e\omega i,$$
- mekanisk motoreffekt
  $$P_m=T_m\omega=k_ti\omega,$$
- friksjonstap
  $$P_f=b\omega^2,$$
- lasteffekt
  $$P_L=T_L\omega.$$

I SI-enheter er $k_e$ og $k_t$ numerisk like for en idealisert motor når enhetene er konsistente. Kontroller energibalansen kvalitativt gjennom startforløpet.

In [ ]:
P_inn = ...
P_R = ...
P_em = ...
P_m = ...
P_f = ...
P_L = ...

# Lag relevante plott og diskuter forskjellen mellom momentan lagring og tap.

# Del D: Automatisk seksjonert start

Vi kombinerer nå ideen fra del A med den dynamiske modellen. Totalmotstanden velges ut fra hastigheten:

$$
R(\omega)=
\begin{cases}
\mathcal R_1,&0\le\omega<\omega_1,\\
\mathcal R_2,&\omega_1\le\omega<\omega_2,\\
\mathcal R_3,&\omega_2\le\omega<\omega_3,\\
R_a,&\omega\ge\omega_3.
\end{cases}
$$

Denne stykkevise modellen er en idealisering. En virkelig starter har brytere, kontaktorer, tidsforsinkelser og vernekretser.

## Oppgave D1: Dynamisk starter

Bruk motorparameterne

$$
V=48\ \mathrm V,\quad L=0.04\ \mathrm H,\quad
R_a=0.40\ \Omega,
$$

$$
k_e=k_t=0.20,\quad J=0.08,
\quad b=0.004,\quad T_L=0.50\ \mathrm{N\,m}.
$$

Implementer den seksjonerte starteren. Ta vare på hvilken motstandstilstand som er aktiv ved hvert tidspunkt.

In [ ]:
V_D = 48.0
L_D = 0.040
R_a_D = R_a
k_e_D = k_e
k_t_D = 0.20
J_D = 0.080
b_D = 0.004
T_L_D = 0.50


def aktiv_motstand(omega):
    if omega < omega_kobling[0]:
        return Rtot1, 1
    elif omega < omega_kobling[1]:
        return Rtot2, 2
    elif omega < omega_kobling[2]:
        return Rtot3, 3
    else:
        return R_a_D, 4


def motor_med_starter(t, x):
    i, omega = x
    R_tot, trinn = aktiv_motstand(omega)

    di = ...
    domega = ...
    return np.array([di, domega])


t_D, X_D = euler_system(
    motor_med_starter,
    x0=np.array([0.0, 0.0]),
    T=8.0,
    h=0.0002
)

i_D = X_D[:, 0]
omega_D = X_D[:, 1]

R_D = np.array([aktiv_motstand(w)[0] for w in omega_D])
trinn_D = np.array([aktiv_motstand(w)[1] for w in omega_D])

In [ ]:
fig, ax = plt.subplots(3, 1, sharex=True, figsize=(8, 9))

ax[0].plot(t_D, i_D)
ax[0].axhline(I_max, color="red", linestyle="--", label="Imax")
ax[0].axhline(I_min, color="orange", linestyle="--", label="Imin")
ax[0].set_ylabel("Strøm A")
ax[0].legend()
ax[0].grid()

ax[1].plot(t_D, omega_D)
ax[1].set_ylabel("Hastighet rad/s")
ax[1].grid()

ax[2].step(t_D, trinn_D, where="post")
ax[2].set_xlabel("Tid s")
ax[2].set_ylabel("Startertrinn")
ax[2].set_yticks([1, 2, 3, 4])
ax[2].grid()

plt.show()

## Oppgave D2: Sammenlign startmetoder

Sammenlign følgende:

1. direkte start med bare $R_a$,
2. én fast startmotstand som kobles ut ved en terskelhastighet,
3. tre seksjonerte startmotstander.

Sammenlign

- maksimal strøm,
- tiden til $95\%$ av slutthastigheten,
- energi tapt i eksterne motstander,
- maksimal akselerasjon,
- hvor brå strømendringene blir ved kobling.

Forklar hvorfor en elektronisk motorstyring kan være mer energieffektiv og gi glattere start enn en resistiv starter.

## Oppgave D3: Ikke-lineær last fra vifte eller pumpe

En enkel modell for en vifte- eller pumpelast er

$$T_L(\omega)=c\omega^2.$$

Erstatt det konstante lastmomentet med denne modellen. Systemet blir da ikke-lineært:

$$
J\dot\omega=k_ti-b\omega-c\omega^2.
$$

Undersøk hvordan sluttfart og oppstartstid påvirkes av $c$.

# Modellkritikk og videreføring

Diskuter minst fire av punktene:

- Magnetfeltet antas konstant.
- Parameterne $R,L,k_e,k_t,J,b$ antas konstante.
- Temperaturøkning og endring i viklingsmotstand er utelatt.
- Børstespenning og kommutering er utelatt.
- Friksjonen er modellert som $b\omega$.
- Lastmomentet er først antatt konstant.
- Bryterne kobler momentant og uten lysbue eller tidsforsinkelse.
- Strømgrenser håndheves bare indirekte gjennom motstandsverdier.
- En moderne elektronisk drift er ikke modellert.

## Mulig videreføring

- temperaturmodell for viklingene,
- elektronisk spenningsstyring,
- regulator for ønsket hastighet,
- generatorvirkning ved negativt moment,
- flerakset mekanisk last,
- kobling til en pumpe eller vifte.

# Oppsummering

Skriv en kort rapport der du forklarer

1. hvorfor startstrømmen blir stor ved stillstand,
2. hvordan det lineære systemet ga motstandsseksjonene,
3. hvordan motspenningen bestemte koblingshastighetene,
4. hvordan den reduserte skalar-ODE-en ble utledet,
5. hva likevektshastigheten og tidskonstanten betyr,
6. hvorfor den fulle modellen er et vanlig ODE-system,
7. hvordan strøm og hastighet utvikler seg på ulike tidsskalaer,
8. hvordan den seksjonerte starteren påvirket strøm, oppstartstid og energitap,
9. hvilke modellforutsetninger som er viktigst.

## Referanser for videre lesning

- Grunnleggende modell for DC-motor: motspenning proporsjonal med hastighet og moment proporsjonalt med strøm.
- Klassiske tre- og firepunktsstartere for DC-maskiner bruker en ekstern motstand delt i seksjoner som gradvis kobles ut.

Studentene trenger ikke lese eksterne kilder for å gjennomføre prosjektet.